# CPE 628 - HW1 Part B: PyTorch (equivalent implementation)

Same two tasks as the TensorFlow notebook, but in PyTorch. The big difference is that PyTorch has no `model.fit()`, so the training loop (forward pass, loss, `backward()`, optimizer step) is written out by hand.

> Ran in Google Colab (Runtime -> Run all). I switched the runtime to the T4 GPU because the MNIST tuning part is slow on CPU.

## Setup

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms

# Hardware selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__} | Running on: {device}")

# Seed setting for reproducibility
torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid")

---
# Task 1: Linear Regression (California Housing)

In [ ]:
# 1. Prepare and scale data
housing = fetch_california_housing()
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Split train set for a validation split (80% train, 20% validation)
val_size = int(0.2 * len(X_train_tensor))
train_size = len(X_train_tensor) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    TensorDataset(X_train_tensor, y_train_tensor), [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
print("train:", len(train_dataset), "| val:", len(val_dataset), "| test:", len(X_test_tensor))

In [ ]:
# 2. Define single-layer linear model
class LinearRegressionModel(nn.Module):

  def __init__(self, in_features: int):
    super().__init__()
    self.linear = nn.Linear(in_features, 1)

  def forward(self, x):
    return self.linear(x)


lin_model = LinearRegressionModel(in_features=X_train.shape[1]).to(device)
criterion_mse = nn.MSELoss()
optimizer_lin = torch.optim.Adam(lin_model.parameters(), lr=0.01)
print(lin_model)

In [ ]:
# 3. Training and validation loop
epochs_lin = 60
train_losses, val_losses = [], []

for epoch in range(epochs_lin):
  # Training phase
  lin_model.train()
  running_train_loss = 0.0
  for batch_x, batch_y in train_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer_lin.zero_grad()
    predictions = lin_model(batch_x)
    loss = criterion_mse(predictions, batch_y)
    loss.backward()
    optimizer_lin.step()

    running_train_loss += loss.item() * batch_x.size(0)

  epoch_train_loss = running_train_loss / len(train_loader.dataset)
  train_losses.append(epoch_train_loss)

  # Validation phase
  lin_model.eval()
  running_val_loss = 0.0
  with torch.no_grad():
    for batch_x, batch_y in val_loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)
      val_preds = lin_model(batch_x)
      val_loss = criterion_mse(val_preds, batch_y)
      running_val_loss += val_loss.item() * batch_x.size(0)

  epoch_val_loss = running_val_loss / len(val_loader.dataset)
  val_losses.append(epoch_val_loss)

  if (epoch + 1) % 10 == 0:
    print(f"Epoch {epoch+1:02d}/{epochs_lin} - train MSE {epoch_train_loss:.4f} | val MSE {epoch_val_loss:.4f}")

In [ ]:
# 4. Evaluate on test set
lin_model.eval()
with torch.no_grad():
  test_preds = lin_model(X_test_tensor.to(device))
  test_mse = criterion_mse(test_preds, y_test_tensor.to(device)).item()
  test_mae = (
      nn.L1Loss()(test_preds, y_test_tensor.to(device)).item()
  )  # L1Loss = MAE

print(f"\n[PyTorch Linear Model] Test MSE: {test_mse:.4f} | Test MAE: {test_mae:.4f}")

# 5. Plot training trajectory
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train Loss (MSE)")
plt.plot(val_losses, label="Val Loss (MSE)")
plt.title("PyTorch: Housing Linear Regression Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.tight_layout()
plt.show()

### Sanity check against the closed-form answer
Linear regression has an exact solution (ordinary least squares), so we can check if gradient descent actually found it.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

ols = LinearRegression().fit(X_train_scaled, y_train)
ols_pred = ols.predict(X_test_scaled)
print(f"sklearn OLS (exact) -> Test MSE: {mean_squared_error(y_test, ols_pred):.4f} | "
      f"MAE: {mean_absolute_error(y_test, ols_pred):.4f}")
print(f"PyTorch Adam        -> Test MSE: {test_mse:.4f} | MAE: {test_mae:.4f}")

comp = pd.DataFrame({
    "OLS weight": ols.coef_,
    "PyTorch weight": lin_model.linear.weight.detach().cpu().numpy().flatten(),
}, index=housing.feature_names)
comp.round(3)

## Task 1 Tuning (PyTorch)

Wrapped the loop above into a function so I can sweep the learning rate and also try plain SGD instead of Adam.

In [ ]:
def train_regressor(model, opt, epochs=60, batch_size=64):
  torch.manual_seed(42)
  tl = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  vl = DataLoader(val_dataset, batch_size=256)
  val_hist = []
  for epoch in range(epochs):
    model.train()
    for bx, by in tl:
      bx, by = bx.to(device), by.to(device)
      opt.zero_grad()
      loss = criterion_mse(model(bx), by)
      loss.backward()
      opt.step()
    model.eval()
    with torch.no_grad():
      tot = sum(criterion_mse(model(bx.to(device)), by.to(device)).item() * bx.size(0) for bx, by in vl)
    val_hist.append(tot / len(val_dataset))
  with torch.no_grad():
    p = model(X_test_tensor.to(device))
    mse = criterion_mse(p, y_test_tensor.to(device)).item()
    mae = nn.L1Loss()(p, y_test_tensor.to(device)).item()
  return val_hist, mse, mae


reg_results, reg_hist = [], {}
for opt_name, lr in [("Adam", 0.001), ("Adam", 0.01), ("Adam", 0.1), ("SGD", 0.01)]:
  torch.manual_seed(42)
  m = LinearRegressionModel(X_train.shape[1]).to(device)
  opt = (torch.optim.Adam if opt_name == "Adam" else torch.optim.SGD)(m.parameters(), lr=lr)
  vh, mse, mae = train_regressor(m, opt)
  key = f"{opt_name} lr={lr}"
  reg_hist[key] = vh
  reg_results.append({"optimizer": opt_name, "lr": lr, "test_MSE": mse, "test_MAE": mae})
  print(f"{key:<14} -> test MSE {mse:.4f} | MAE {mae:.4f}")

plt.figure(figsize=(8, 4))
for k, v in reg_hist.items():
  plt.plot(v, label=k)
plt.yscale("log")
plt.title("PyTorch linear regression: val MSE by optimizer / lr")
plt.xlabel("Epoch")
plt.ylabel("Val MSE (log)")
plt.legend()
plt.tight_layout()
plt.show()

pd.DataFrame(reg_results)

---
# Task 2: MNIST Classification

In [ ]:
# 1. Download and normalize MNIST data
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL image to Tensor in range [0.0, 1.0]
])

train_data_mnist = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_data_mnist = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader_mnist = DataLoader(
    train_data_mnist, batch_size=128, shuffle=True, pin_memory=True
)
test_loader_mnist = DataLoader(
    test_data_mnist, batch_size=128, shuffle=False, pin_memory=True
)

# peek at a few samples
plt.figure(figsize=(6, 6))
for i in range(9):
  img, label = train_data_mnist[i]
  plt.subplot(3, 3, i + 1)
  plt.imshow(img.squeeze(), cmap="gray")
  plt.title(f"Label: {label}")
  plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Define multi-layer perceptron architecture
class MNISTClassifier(nn.Module):

  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 10),  # Outputs raw logits
    )

  def forward(self, x):
    return self.network(x)


mnist_clf = MNISTClassifier().to(device)
criterion_ce = (
    nn.CrossEntropyLoss()
)  # Combines LogSoftmax and NLLLoss numerically
optimizer_mnist = torch.optim.Adam(mnist_clf.parameters(), lr=1e-3)
print(mnist_clf)
print("trainable params:", sum(p.numel() for p in mnist_clf.parameters() if p.requires_grad))

In [ ]:
# 3. Train and evaluate loop
epochs_mnist = 15
history_acc, history_loss = [], []
val_acc_list, val_loss_list = [], []
start = time.time()

for epoch in range(epochs_mnist):
  # Training
  mnist_clf.train()
  total_loss, correct, total = 0.0, 0, 0
  for images, labels in train_loader_mnist:
    images, labels = images.to(device), labels.to(device)

    optimizer_mnist.zero_grad()
    outputs = mnist_clf(images)
    loss = criterion_ce(outputs, labels)
    loss.backward()
    optimizer_mnist.step()

    total_loss += loss.item() * images.size(0)
    _, predicted = torch.max(outputs, 1)
    correct += (predicted == labels).sum().item()
    total += labels.size(0)

  epoch_train_loss = total_loss / total
  epoch_train_acc = correct / total
  history_loss.append(epoch_train_loss)
  history_acc.append(epoch_train_acc)

  # Validation / Testing
  mnist_clf.eval()
  v_loss, v_correct, v_total = 0.0, 0, 0
  with torch.no_grad():
    for images, labels in test_loader_mnist:
      images, labels = images.to(device), labels.to(device)
      outputs = mnist_clf(images)
      loss = criterion_ce(outputs, labels)

      v_loss += loss.item() * images.size(0)
      _, predicted = torch.max(outputs, 1)
      v_correct += (predicted == labels).sum().item()
      v_total += labels.size(0)

  epoch_val_loss = v_loss / v_total
  epoch_val_acc = v_correct / v_total
  val_loss_list.append(epoch_val_loss)
  val_acc_list.append(epoch_val_acc)

  print(
      f"Epoch [{epoch+1:02d}/{epochs_mnist:02d}] - "
      f"Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc*100:.2f}% | "
      f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%"
  )

print(f"\n[PyTorch MNIST Model] Final Test Accuracy: {val_acc_list[-1]*100:.2f}%")
print(f"Training time: {time.time() - start:.1f}s")

In [ ]:
# 4. Plot training and test curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_acc, label="Train Acc")
axes[0].plot(val_acc_list, label="Test Acc")
axes[0].set_title("PyTorch: Accuracy Progression")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history_loss, label="Train Loss")
axes[1].plot(val_loss_list, label="Test Loss")
axes[1].set_title("PyTorch: Cross-Entropy Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## Task 2 Tuning (PyTorch)

Same idea as the TF notebook: hold out 6,000 training images as a real validation set, 10 epochs per run, change one thing at a time. Here I try dropout, learning rate, and swapping Adam for SGD with momentum.

In [ ]:
gen = torch.Generator().manual_seed(42)
tr_sub, va_sub = torch.utils.data.random_split(train_data_mnist, [54000, 6000], generator=gen)


def make_mlp(dropout=0.2):
  layers_ = [nn.Flatten()]
  for i, o in [(784, 512), (512, 256), (256, 128)]:
    layers_ += [nn.Linear(i, o), nn.ReLU()]
    if dropout > 0:
      layers_.append(nn.Dropout(dropout))
  layers_.append(nn.Linear(128, 10))
  return nn.Sequential(*layers_)


def accuracy_and_loss(model, loader):
  model.eval()
  loss_sum, correct, n = 0.0, 0, 0
  with torch.no_grad():
    for x, t in loader:
      x, t = x.to(device), t.to(device)
      out = model(x)
      loss_sum += criterion_ce(out, t).item() * x.size(0)
      correct += (out.argmax(1) == t).sum().item()
      n += x.size(0)
  return correct / n, loss_sum / n


def run_config(dropout=0.2, lr=1e-3, opt_name="Adam", epochs=10, batch_size=128):
  torch.manual_seed(42)
  model = make_mlp(dropout).to(device)
  if opt_name == "Adam":
    opt = torch.optim.Adam(model.parameters(), lr=lr)
  else:
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
  tl = DataLoader(tr_sub, batch_size=batch_size, shuffle=True, pin_memory=True)
  vl = DataLoader(va_sub, batch_size=512)
  hist = {"train_loss": [], "val_loss": [], "val_acc": []}
  for epoch in range(epochs):
    model.train()
    tot, n = 0.0, 0
    for x, t in tl:
      x, t = x.to(device), t.to(device)
      opt.zero_grad()
      loss = criterion_ce(model(x), t)
      loss.backward()
      opt.step()
      tot += loss.item() * x.size(0)
      n += x.size(0)
    va, vloss = accuracy_and_loss(model, vl)
    hist["train_loss"].append(tot / n)
    hist["val_loss"].append(vloss)
    hist["val_acc"].append(va)
  test_acc, _ = accuracy_and_loss(model, test_loader_mnist)
  return hist, test_acc


pt_configs = [
    ("baseline",         dict(dropout=0.2, lr=1e-3, opt_name="Adam")),
    ("no dropout",       dict(dropout=0.0, lr=1e-3, opt_name="Adam")),
    ("dropout=0.5",      dict(dropout=0.5, lr=1e-3, opt_name="Adam")),
    ("Adam lr=1e-2",     dict(dropout=0.2, lr=1e-2, opt_name="Adam")),
    ("Adam lr=1e-4",     dict(dropout=0.2, lr=1e-4, opt_name="Adam")),
    ("SGD+mom lr=0.05",  dict(dropout=0.2, lr=0.05, opt_name="SGD")),
]

pt_results, pt_hist = [], {}
for name, kw in pt_configs:
  t0 = time.time()
  h, t_acc = run_config(**kw)
  pt_hist[name] = h
  pt_results.append({"config": name, "final_val_acc": h["val_acc"][-1],
                     "best_val_acc": max(h["val_acc"]), "test_acc": t_acc,
                     "val_loss_min_epoch": int(np.argmin(h["val_loss"])) + 1,
                     "seconds": round(time.time() - t0, 1)})
  print(f"{name:<16} val acc {h['val_acc'][-1]*100:.2f}% | test acc {t_acc*100:.2f}%")

pd.DataFrame(pt_results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, h in pt_hist.items():
  axes[0].plot(h["val_acc"], label=name)
  axes[1].plot(h["val_loss"], label=name)
axes[0].set_title("PyTorch: val accuracy by config")
axes[0].set_xlabel("Epoch")
axes[0].set_ylim(0.95, 1.0)
axes[1].set_title("PyTorch: val loss by config")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 0.2)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()